In [1]:
from flask import Flask, request, jsonify
from tensorflow.keras.models import load_model
import os
import numpy as np
from io import BytesIO
from PIL import Image

In [1]:
app = Flask(__name__)

# Define base directory for model paths
BASE_DIR = ../

# Mapping of plant types to their respective model file paths
MODEL_PATHS = {
    "apple": os.path.join(BASE_DIR, "Apple-disease", "models", "model_version_v1.keras"),
    "banana": os.path.join(BASE_DIR, "Banana-disease", "models", "model_version_v1.keras"),
    "beans": os.path.join(BASE_DIR, "Beans-disease", "models", "model_version_v1.keras"),
    "cassava": os.path.join(BASE_DIR, "Cassava-disease", "models", "model_version_v1.keras"),
    "grape": os.path.join(BASE_DIR, "Grape-disease", "models", "model_version_v1.keras"),
    "groundnut": os.path.join(BASE_DIR, "Groundnut-disease", "models", "model_version_v1.keras"),
    "maize": os.path.join(BASE_DIR, "Maize (Corn)-disease", "models", "model_version_v1.keras"),
    "potato": os.path.join(BASE_DIR, "potato-disease", "models", "model_version_v1.keras"),
    "rice": os.path.join(BASE_DIR, "rice-disease", "models", "model_version_v1.keras"),
    "tomatoes": os.path.join(BASE_DIR, "Tomatoes-disease", "models", "model_version_v1.keras"),
}

# Define class names for each plant disease type
# This dictionary maps plant types to their possible disease categories
class_names_dict = {
    'apple': [
        'Apple___alternaria_leaf_spot',
        'Apple___black_rot',
        'Apple___brown_spot',
        'Apple___gray_spot',
        'Apple___healthy',
        'Apple___rust',
        'Apple___scab'
    ],
    'banana': [
        'Bract_Mosaic_Virus',
        'Fungal_Infection',
        'Healthy',
        'Insect_Damage',
        'Moko_Disease',
        'Panama_Disease',
        'Sigatoka_Disease'
    ],
    'beans': [
        'angular_leaf_spot', 
        'bean_rust', 
        'healthy'
    ],
    'cassava': [
        'Cassava___bacterial_blight',
        'Cassava___brown_streak_disease',
        'Cassava___green_mottle',
        'Cassava___healthy',
        'Cassava___mosaic_disease',
    ],
    'grape': [
        'Grape___Black_rot',
        'Grape___Esca_(Black_Measles)',
        'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)',
        'Grape___healthy'
    ],
    'groundnut': [
        'early_leaf_spot',
        'healthy leaf',
        'late leaf spot',
        'nutrition deficiency',
        'rust'
    ],
    'maize': [
        'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
        'Corn_(maize)___Common_rust_',
        'Corn_(maize)___Northern_Leaf_Blight',
        'Corn_(maize)___healthy'
    ],
    'potato': [
        'Potato___Early_blight', 
        'Potato___Late_blight', 
        'Potato___healthy'
    ],
    'rice': [
        'Rice_Brown_spot', 
        'Rice_Healthy', 
        'Rice_Hispa', 
        'Rice_Leaf_blast'
    ],
    'tomatoes': [
        'Tomato___Bacterial_spot',
        'Tomato___Early_blight',
        'Tomato___Late_blight',
        'Tomato___Leaf_Mold',
        'Tomato___Septoria_leaf_spot',
        'Tomato___Spider_mites Two-spotted_spider_mite',
        'Tomato___Target_Spot',
        'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
        'Tomato___Tomato_mosaic_virus',
        'Tomato___healthy'
    ],
}

# Load all models into memory
try:
    # Initialize a dictionary to store loaded models
    models = {plant: load_model(path) for plant, path in MODEL_PATHS.items()}
    print("Models loaded successfully!")
except Exception as e:
    print(f"Error loading models: {e}")

# Function to preprocess image data
# Converts uploaded image data into a format suitable for model prediction
def read_file_as_image(file_data) -> np.ndarray:
    image = np.array(Image.open(BytesIO(file_data)))
    return image

# Define the prediction endpoint
@app.route('/predict', methods=['POST'])
def predict():
    try:
        # Extract plant type and uploaded file from the request
        plant = request.form.get('plant', '').lower()
        file = request.files.get('file')

        # Validate the plant type
        if plant not in models:
            return jsonify({"error": f"Invalid plant type. Available options: {list(models.keys())}"})

        # Ensure a file is provided
        if not file:
            return jsonify({"error": "No file provided."})

        # Read the uploaded file's content
        file_data = file.read()

        # Get the corresponding class names for the plant
        class_names = class_names_dict[plant]

        # Preprocess the image for prediction
        image = read_file_as_image(file_data)
        img_batch = np.expand_dims(image, 0)  # Add batch dimension

        # Perform prediction using the relevant model
        predictions = models[plant].predict(img_batch)
        result_index = np.argmax(predictions[0])  # Get index of highest confidence
        result_label = class_names[result_index]  # Get label corresponding to the index
        confidence = float(np.max(predictions[0]))  # Get confidence score

        # Return prediction results as JSON
        return jsonify({"plant": plant, "predictions": result_label, "confidence": confidence})
    except Exception as e:
        # Handle errors and return them as JSON
        return jsonify({"error": str(e)})

# Run the Flask application
if __name__ == '__main__':
    app.run(debug=True, use_reloader=False)


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with watchdog (windowsapi)


SystemExit: 1

C:\Users\DELL\anaconda3\envs\env_gpu\lib\site-packages\IPython\core\interactiveshell.py:3534: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
